In [ ]:
import glob
import os

import numpy as np
import pandas as pd

# Load the dataset
path = r"../datasets/cicids2017"
all_files = glob.glob(os.path.join(path, "*.csv"))
df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

# Remove leading and trailing whitespace from column names
df.columns = df.columns.str.strip()

# Display the dataframe info
df.info()

As we can see, there are 79 columns in the dataset. However, there is a duplicate column named `Fwd Header Length.1` which is likely a duplicate of `Fwd Header Length`. We will check if they are identical and then drop the duplicate column if they are indeed the same.

In [ ]:
# Check if the duplicate columns are identical, and if so, drop the duplicate column
if df["Fwd Header Length"].equals(df["Fwd Header Length.1"]):
    print(
        "The columns 'Fwd Header Length' and 'Fwd Header Length.1' are identical. Proceeding to drop the duplicate column."
    )
    df.drop(columns=["Fwd Header Length.1"], inplace=True)
else:
    print("The columns 'Fwd Header Length' and 'Fwd Header Length.1' are not identical.")

Display the dataframe info again to confirm that the duplicate column has been dropped (if they were identical).

In [ ]:
df.info()

We will convert all columns (except `Label` as it contains categorical data) to numeric values, coercing any errors to NaN. This will help us identify any non-numeric values that may be present in the dataset.

In [ ]:
labels = df["Label"]

# Convert all columns (except 'Label') to numeric, coercing errors to NaN
df = df.drop(columns=["Label"]).apply(pd.to_numeric, errors="coerce")

# Reattach the 'Label' column to the dataframe
df["Label"] = labels

Replace positive and negative infinity with missing values, then inspect missing values in each column.

In [ ]:
# Convert infinite feature values to missing values
feature_columns = df.columns.drop("Label")
df[feature_columns] = df[feature_columns].replace([np.inf, -np.inf], np.nan)

# Check for missing values
missing_values = df.isna().sum()

# Display columns with missing values
missing_values[missing_values > 0]

Drop rows with missing values in any feature so the saved dataset contains only finite numeric inputs for modeling.

In [ ]:
# Drop rows with missing values in any feature
df.dropna(subset=feature_columns, inplace=True)

# Check for missing values again to confirm that they have been removed
missing_values = df.isna().sum()
missing_values[missing_values > 0]

Now that we have cleaned the dataset by removing duplicate columns and handling missing values, we will encode the `Label` column using Label Encoding. This will convert the categorical labels into numeric values, which can be used for machine learning models.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Encode the 'Label' column using Label Encoding
df["Label"] = le.fit_transform(df["Label"])

# Create a mapping of the original labels to their encoded values
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

# Display the mapping of original labels to encoded values
print("Label Mapping:")
for original_label, encoded_value in label_mapping.items():
    print(f"\t{original_label}: {encoded_value}")

Lastly, we will save the cleaned dataset to a new CSV file called `cicids2017_cleaned.csv` for future use in our analysis and modeling. Also, we will save the label mapping to a JSON file called `label_mapping.json` to keep track of the original labels and their corresponding encoded values.

In [ ]:
# Save the cleaned dataset to a new CSV file
df.to_csv("../datasets/cicids2017_cleaned.csv", index=False)

import json

# Convert the label mapping to a JSON-serializable format and save it to a JSON file
label_mapping = {key: int(value) for key, value in label_mapping.items()}

# Save the label mapping to a JSON file
with open("../datasets/cicids2017_label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)